# ViSpatialQA — Bước 1 (phân tích lỗi Human) + Bước 2 (stacking)

**Chỉ cần CPU:** không cài vLLM, không nạp model nào. Notebook đọc các dự đoán ngoài mẫu mà E4 đã lưu trên Drive (`outputs/predictions_e4`), trên 613 câu Human (train + dev).

- **Bước 1 — `analyze_errors.py`:** ensemble sai ở đâu và vì sao.
  - Mỗi loại câu sai bao nhiêu, và bao nhiêu câu trong đó có ít nhất một nguồn đúng (cứu được bằng cách gộp tốt hơn) so với không nguồn nào đúng (cần ý tưởng mới, hoặc gold mâu thuẫn với câu chuyện).
  - Kiểu lỗi FR: sai hướng, thiếu gần/xa, thừa gần/xa, đổi gần↔xa, chạm, DK.
  - Ma trận nhầm lẫn YN và CO; điểm theo `reasoning_type`.
  - LPX sai do bỏ phiếu hay chép không ổn định (có cặp world/dạng câu hỏi đúng), hay mọi cặp đều sai.
  - `errors.csv`: mọi câu sai kèm câu chuyện, gold và đáp án từng nguồn, để soi tay.
- **Bước 2 — `stack_ensemble.py`:** thay trọng số cố định bằng logistic regression cho từng loại câu. Đặc trưng là tỷ lệ phiếu, nguồn có trả lời không, số nguồn đồng ý. Đánh giá trên **cùng các fold story** với cv của ensemble hiện tại (0.846) để so trực tiếp, rồi ghi file nộp `stack` và `hybrid`.

**Thời gian ước tính:** chuẩn bị 1–2 phút (clone, chép dữ liệu từ Drive), bước 1 dưới 1 phút, bước 2 dưới 1 phút. Tổng khoảng 3–5 phút. Chọn runtime CPU là đủ.

## 1. CONFIG

In [ ]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/e5-analysis-stacking",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    BRANCHES="F=F2C,F1 S=S L=LCOT P=LPX",           # tổ hợp tốt nhất của E4
    BEST_E4_DIR="e4/trdev_LPX/F+S+L+P",              # lấy auto_submission.json từ đây cho đủ cặp file nộp
    C_GRID="0.1,1,10",
)

## 2. Chuẩn bị: Drive, code, dữ liệu, dự đoán E4

In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import Markdown, display

C = CONFIG
WORK, DRIVE_OUT = Path(C["WORK_DIR"]), Path(C["DRIVE_BACKUP_DIR"])
START = time.time()


def sh(cmd):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=WORK, text=True, capture_output=True)
    print(result.stdout[-20000:], result.stderr[-5000:] if result.returncode else "", flush=True)
    if result.returncode:
        raise RuntimeError(f"Lệnh lỗi (exit {result.returncode})")


if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"], check=True)

for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/human_dev.json").exists():
    sh(["python", "make_splits.py", "--from-manifest"])

PRED = WORK / "outputs/predictions_e4"
shutil.copytree(DRIVE_OUT / "predictions_e4", PRED, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("*.raw.jsonl", "F2C_folds"))
E4 = WORK / "outputs/e4"
E4.mkdir(parents=True, exist_ok=True)
for name in ("lp_choice.json",):
    shutil.copy2(DRIVE_OUT / "e4" / name, E4 / name)
auto_sub = DRIVE_OUT / C["BEST_E4_DIR"] / "auto_submission.json"
need = ["F2C/human_train.jsonl", "F1/human_train.jsonl", "LCOT/human_train.jsonl", "LPX/human_train.jsonl",
        "LP/human_train.parses.jsonl", "LPX/human_dev.jsonl", "LPX/human_test.jsonl"]
missing = [n for n in need if not (PRED / n).exists()]
assert not missing, f"Thiếu {missing} trong Drive outputs/predictions_e4: chạy run_colab_e4.ipynb trước."
sh(["python", "crossfit.py", "combine", "--pred-dir", PRED, "--name", "trdev"])
print(f"Chuẩn bị xong sau {(time.time() - START) / 60:.1f} phút")

## 3. Bước 1: phân tích lỗi Human (613 câu, ensemble ngoài fold)
Số "Ensemble" ở đây phải khớp cv của E4 (Human 0.846); nếu khác thì các nguồn đã bị thay đổi.

In [ ]:
OUT = WORK / "outputs/e5"
t0 = time.time()
sh(["python", "analyze_errors.py", "--pred-dir", PRED, "--split", "trdev", "--branches", C["BRANCHES"],
    "--lp-source", "LPX", "--lp-settings", E4 / "lp_choice.json", "--output-dir", OUT / "analysis"])
display(Markdown((OUT / "analysis/report.md").read_text(encoding="utf-8")))
print(f"Bước 1 xong sau {(time.time() - t0) / 60:.1f} phút. Mọi câu sai: {OUT / 'analysis/errors.csv'}")

In [ ]:
import pandas as pd
errors = pd.read_csv(OUT / "analysis/errors.csv")
print(len(errors), "câu sai")
print("\nCâu không nguồn nào đúng (ứng viên gold mâu thuẫn / cần ý tưởng mới):")
display(errors[errors["sources_right"] == "không"][["key", "q_type", "question", "gold", "ensemble", "fr_kind", "lp_diagnosis"]])

## 4. Bước 2: stacking so với ensemble trọng số (cùng fold)
Nếu stacking hoặc hybrid có Human cv cao hơn 0.846 thì file nộp mới nằm ở `outputs/e5/stacking/{stack,hybrid}/`.

In [ ]:
t0 = time.time()
sh(["python", "stack_ensemble.py", "--pred-dir", PRED, "--split", "trdev", "--branches", C["BRANCHES"],
    "--c-grid", C["C_GRID"], "--auto-submission", auto_sub, "--output-dir", OUT / "stacking"])
display(Markdown((OUT / "stacking/report.md").read_text(encoding="utf-8")))
shutil.copytree(OUT, DRIVE_OUT / "e5", dirs_exist_ok=True)
print(f"Bước 2 xong sau {(time.time() - t0) / 60:.1f} phút. Tổng {(time.time() - START) / 60:.1f} phút. "
      f"Đã sao lưu {OUT} -> {DRIVE_OUT / 'e5'}")